## [SITCOM-1768] - Look for correlations between fa following errors and the commands they were given

Given a certain time range, this notebook will compute the delays between M1M3 appliedCylinderForces in primary and secondary axes of the force actuators (FAs) and the measured forces in them, and visualize the result per FA in milliseconds.

[SITCOM-1768]: https://rubinobs.atlassian.net/browse/SITCOM-1768

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd

from astropy.time import Time
from scipy.signal import correlate, correlation_lags
import matplotlib.pyplot as plt

from lsst.summit.utils.efdUtils import EfdClient, getEfdData, makeEfdClient
from lsst.sitcom.vandv import m1m3
from lsst.ts.xml.tables.m1m3 import FATable

In [ ]:
N_PRIMARY = 156 # this is the number of z-axis actuators
N_SECONDARY = 112 # this is the number of x-y axis actuators, that appear in dual axes ones
MEASUREMENT_INTERVAL = 20 # in ms, currently hard-coded, corresponds to 50 Hz

In [ ]:
#this function is used for visualization of the time delayed 
# target force signal
def displace_and_pad(array, n):
    # Ensure n is a positive integer
    if n < 0:
        raise ValueError("n must be a non-negative integer")
    
    # Create a new array with the same initial value padded at the beginning
    initial_value = array[0]
    padded_array = np.full(n, initial_value)
    
    # Concatenate the padded array with the original array, excluding the last n elements
    result = np.concatenate((padded_array, array[:-n]))
    
    return result

In [ ]:
def heat_map_z(primary,secondary, axp, axs, zmin, zmax): 
    # adapted from Craig's notebook for SITCOMTN-107
    # This builds the following error heat maps
    types = [['SAA','NA', 'o', 'Z'], \
             ['DAA','Y_PLUS', '^', 'Y_PLUS'], \
             ['DAA','Y_MINUS', 'v', 'Y_MINUS'], \
             ['DAA','X_PLUS', '>', 'X_PLUS'], \
             ['DAA','X_MINUS', '<', 'X_MINUS']]
    axp.set_title("Primary")
    axp.set_xlabel("X position (m)")
    axp.set_ylabel("Y position (m)")
    axp.set_xlim(-5,5)
    axp.set_ylim(-5,5)

    for [type, orient, marker, label] in types:
        xs = []
        ys = []
        zs = []
        for i in range(len(FATable)):
            x = FATable[i].x_position
            y = FATable[i].y_position
            if FATable[i].actuator_type.name == type and \
                FATable[i].orientation.name == orient:
                xs.append(x)
                ys.append(y)
                zs.append(primary[i])
        im = axp.scatter(xs, ys, marker='o', c=zs, cmap='hot', \
                         vmin=zmin, vmax=zmax, s=50, label=label)
    plt.colorbar(im, ax=axp,fraction=0.055, pad=0.02, cmap='hot')  

    axs.set_title("Secondary")
    axs.set_xlabel("X position (m)")
    axs.set_xlim(-5,5)
    axs.set_ylim(-5,5)
    for [type, orient, marker, label] in types:
        if type == 'SAA':
            continue
        xs = []
        ys = []
        zs = []
        for i in range(len(FATable)):
            x = FATable[i].x_position
            y = FATable[i].y_position
            if FATable[i].actuator_type.name == type and \
                FATable[i].orientation.name == orient:
                xs.append(x)
                ys.append(y)
                zs.append(secondary[FATable[i].s_index])
        im = axs.scatter(xs, ys, marker=marker, c=zs, cmap='hot', \
                         vmin=zmin, vmax=zmax, s=50, label=label)
    plt.colorbar(im, ax=axs,fraction=0.055, pad=0.02, cmap='hot')  


In [ ]:
topic_target = f"lsst.sal.MTM1M3.appliedCylinderForces"
topic_measured = f"lsst.sal.MTM1M3.forceActuatorData"

In [ ]:
client = makeEfdClient()

In [ ]:
#define start and end time to evaluate delays, tested with a 10 s period
# around a test case at Time("2024-12-12 09:20:34Z", scale="utc")
start = Time("2024-12-12 09:20:34Z", scale="utc")
end = Time("2024-12-12 09:20:44Z", scale="utc")

In [ ]:
#retrieve target and measured forces in primary and secondary axes of all FAs
primary_target_cylinder_force = [f"primaryCylinderForces{i}" for i in range(N_PRIMARY)]
df_primary_target_cylinder_force = getEfdData(
    client, topic_target, columns=primary_target_cylinder_force, begin=start, end=end
) 
primary_measured_cylinder_force = [f"primaryCylinderForce{i}" for i in range(N_PRIMARY)]
df_primary_measured_cylinder_force = getEfdData(
    client, topic_measured, columns=primary_measured_cylinder_force, begin=start, end=end
) 
secondary_target_cylinder_force = [f"secondaryCylinderForces{i}" for i in range(N_SECONDARY)]
df_secondary_target_cylinder_force = getEfdData(
    client, topic_target, columns=secondary_target_cylinder_force, begin=start, end=end
) 
secondary_measured_cylinder_force = [f"secondaryCylinderForce{i}" for i in range(N_SECONDARY)]
df_secondary_measured_cylinder_force = getEfdData(
    client, topic_measured, columns=secondary_measured_cylinder_force, begin=start, end=end
) 

In [ ]:
# compute delay between target and measured separately for each actuator and axis
# through a cross-correlation
# it is important to span a 'sufficiently' wide time range, otherwise the math will
# make the delay to be zero
# ALTERNATIVE to be tested: use displace_and_pad function and compute the residual at each point of the interval
#  between signal1 and signal2. Choose the displacement in ms which leads to the smallest RMS. 
delay_primary = np.empty(N_PRIMARY)
delay_ms_primary = np.empty(N_PRIMARY)
delay_secondary = np.empty(N_SECONDARY)
delay_ms_secondary = np.empty(N_SECONDARY)
signal1_primary = np.empty(N_PRIMARY, dtype=object)
signal2_primary = np.empty(N_PRIMARY, dtype=object)
signal1_secondary = np.empty(N_SECONDARY, dtype=object)
signal2_secondary = np.empty(N_SECONDARY, dtype=object)
for i in range(N_PRIMARY):
    meas_primary = df_primary_measured_cylinder_force[f'primaryCylinderForce{i}'].values
    targ_primary = df_primary_target_cylinder_force[f'primaryCylinderForces{i}'].values
    targ_primary = targ_primary/1000.
    signal1_primary[i] = meas_primary - np.mean(meas_primary)
    signal2_primary[i] = targ_primary - np.mean(targ_primary)
    #see the discussion here
    #https://stackoverflow.com/questions/69117617/how-to-find-the-lag-between-two-time-series-using-cross-correlation
    #https://stackoverflow.com/questions/72230482/signal-correlation-shift-and-lag-correct-only-if-arrays-subtracted-by-mean
    cross_correlation = correlate(signal1_primary[i], signal2_primary[i], mode='full')
    lags = correlation_lags(len(signal1_primary[i]), len(signal2_primary[i]), mode='full')
    delay_primary[i] = lags[np.argmax(cross_correlation)]
    delay_ms_primary[i] = delay_primary[i] * MEASUREMENT_INTERVAL
    #print(i,delay_ms_primary[i])

print(f"Average delay for primary (Z) axes is {np.mean(delay_ms_primary)} ms")

for i in range(N_SECONDARY):
    meas_secondary = df_secondary_measured_cylinder_force[f'secondaryCylinderForce{i}'].values
    targ_secondary = df_secondary_target_cylinder_force[f'secondaryCylinderForces{i}'].values
    targ_secondary = targ_secondary/1000.
    signal1_secondary[i] = meas_secondary - np.mean(meas_secondary)
    signal2_secondary[i] = targ_secondary - np.mean(targ_secondary)
    #see the discussion here
    #https://stackoverflow.com/questions/69117617/how-to-find-the-lag-between-two-time-series-using-cross-correlation
    #https://stackoverflow.com/questions/72230482/signal-correlation-shift-and-lag-correct-only-if-arrays-subtracted-by-mean
    cross_correlation = correlate(signal1_secondary[i], signal2_secondary[i], mode='full')
    lags = correlation_lags(len(signal1_secondary[i]), len(signal2_secondary[i]), mode='full')
    delay_secondary[i] = lags[np.argmax(cross_correlation)]
    delay_ms_secondary[i] = delay_secondary[i] * MEASUREMENT_INTERVAL
    #print(i,delay_ms_secondary[i])

print(f"Average delay for secondary axes is {np.mean(delay_ms_secondary)} ms")

In [ ]:
#this cell will plot a single actuator time evolution
# uncomment the prints in the previous cell to see delays measured at each FA, and use the index here
actuator = 3 # use index printed in previous cell
fig, ax = plt.subplots()
ax.plot(signal1_primary[actuator], label='FA data') #you can also use secondary
ax.plot(signal2_primary[actuator], label='target', linestyle='dashed')
# now we can add the target signal displaced by the computed delay, to check if this delay seems reasonable
signal_displaced = displace_and_pad(signal2_primary[actuator], int(delay_primary[actuator]))
print(f"Delay is {delay_primary[actuator]} measurements which equates to {delay_ms_primary[actuator]} ms")
ax.plot(signal_displaced, label='displaced signal', color='orange')
plt.ylabel(f'Primary Cylinder Force {actuator} (- mean)')
plt.xlabel('Measurement number')
delay_text = str(delay_ms_primary[actuator])
ax.text(0.1, 0.5, f'Delay: {delay_text} ms',transform=ax.transAxes)
fig.legend()

In [ ]:
# Plot the snapshot
zmin = 0
zmax = 200.0

fig = plt.figure(figsize=(8,8))
fig.suptitle(f"FA (measured - target) delays (ms); {start}", x=0.5, y=0.85)
axp = fig.add_axes((0.1, 0.45, 0.35, 0.35))
axs = fig.add_axes((0.55, 0.45, 0.35, 0.35))
heat_map_z(delay_ms_primary, delay_ms_secondary, axp, axs, zmin, zmax)